In [0]:
from pyspark.sql.functions import *

df_bronze = spark.read.csv('/Volumes/kautuk_catalog/csv_to_table/data/titanic/titanic.csv', header=True, inferSchema=True)
# display(df_bronze)

df_bronze.filter("Survived = 1").write.mode("overwrite").saveAsTable("kautuk_catalog.csv_to_table.titanic_survivors_bronze")

df_titanic_survivors = spark.read.table("kautuk_catalog.csv_to_table.titanic_survivors_bronze")

df_titanic_survivors = df_titanic_survivors.withColumn('Age', coalesce(col('Age').try_cast('Integer')))\
    .withColumn("Name",trim(regexp_replace(regexp_replace(col("Name"), r'["()]', ""), r'\s+', ' ')))\
    .withColumn("First_Name", split(col("Name"), ',')[1])\
    .withColumn("Last_Name", split(col("Name"), ',')[0])\
    .withColumn("First_Name", split(col("First_Name"), r'\.')[1])\
    .withColumn("Name", concat_ws('', col("Last_Name"), col("First_Name")))\
    .drop("First_Name", "Last_Name")

string_cols = [c for c, t in df_titanic_survivors.dtypes if t == "string"]
df_titanic_survivors = df_titanic_survivors.select(
    *[
        when(col(c) == r"\N", None).otherwise(col(c)).alias(c) if c in string_cols else col(c)
        for c in df_titanic_survivors.columns
    ]
)

# display(df_titanic_survivors)

df_titanic_survivors.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("kautuk_catalog.csv_to_table.titanic_survivors_silver")

# .withColumn("Name",trim(regexp_replace(regexp_replace(col("Name"), r'["()]', ""), r'\s+', ' ')))\
        
# df_titanic_survivors_clean = df_titanic_survivors.withColumn(
#     "Name",
#     trim(regexp_replace(col("Name"), r'(\s*["(][^")]*[")]\s*)', ''))
# )        

# df_titanic_survivors_incorrect_names = df_titanic_survivors.filter("""Name like '%)%' or Name like '%"%'""")
# display(df_titanic_survivors_incorrect_names)


# from pyspark.sql.functions import regexp_replace, trim
# df_titanic_survivors_clean = df_titanic_survivors.withColumn(
#     "Name",
#     trim(regexp_replace(col("Name"), r'(\s*["(][^")]*[")]\s*)', ''))
# )
# display(df_titanic_survivors_clean)

In [0]:
%sql
select * from kautuk_catalog.csv_to_table.titanic_survivors_bronze

In [0]:
%sql
select * from kautuk_catalog.csv_to_table.titanic_survivors_silver

In [0]:
from pyspark.sql.functions import count, col

# Count male and female survivors
df_gender_report = df_titanic_survivors.groupBy("Sex").agg(count("*").alias("Survivor_Count"))
display(df_gender_report)

# Plot gender survivor count
import matplotlib.pyplot as plt

gender_pd = df_gender_report.toPandas()
plt.figure(figsize=(5, 3))
plt.bar(gender_pd['Sex'], gender_pd['Survivor_Count'])
plt.xlabel('Sex')
plt.ylabel('Survivor Count')
plt.title('Titanic Survivors by Gender')
plt.tight_layout()
plt.show()

# Find the safest Pclass (highest survivor count)
df_pclass_report = df_titanic_survivors.groupBy("Pclass").agg(count("*").alias("Survivor_Count")).orderBy(col("Survivor_Count").desc())
display(df_pclass_report)

# Plot Pclass survivor count
pclass_pd = df_pclass_report.toPandas()
plt.figure(figsize=(5, 3))
plt.bar(pclass_pd['Pclass'], pclass_pd['Survivor_Count'])
plt.xlabel('Pclass')
plt.ylabel('Survivor Count')
plt.title('Titanic Survivors by Pclass')
plt.tight_layout()
plt.show()
